# Problem 2: Transformer from Scratch


## 1. Attention from Scratch & Unit Tests


In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Task 1: Scaled Dot-Product Attention from Scratch
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

# Unit Test 1: Check rows sum to 1
Q_t = torch.randn(2, 4, 10, 16)
K_t = torch.randn(2, 4, 10, 16)
V_t = torch.randn(2, 4, 10, 16)
out_t, weights_t = scaled_dot_product_attention(Q_t, K_t, V_t)
row_sums = weights_t.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5)

# Unit Test 2: Match torch.nn.functional.scaled_dot_product_attention
sdpa_torch = F.scaled_dot_product_attention(Q_t, K_t, V_t)
assert torch.allclose(out_t, sdpa_torch, atol=1e-5)

# Unit Test 3: Hand-checkable 3-token example from specification
Q_hand = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]) # (1, 3, 2)
K_hand = torch.tensor([[[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]]])
V_hand = torch.tensor([[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]])

# Create Causal Mask for 3 tokens
causal_mask = torch.tril(torch.ones(3, 3)).unsqueeze(0).unsqueeze(0)
out_hand, weights_hand = scaled_dot_product_attention(
    Q_hand.unsqueeze(1), K_hand.unsqueeze(1), V_hand.unsqueeze(1), mask=causal_mask
)

expected_weights = torch.tensor([
    [1.0000, 0.0000, 0.0000],
    [0.3299, 0.6701, 0.0000],
    [0.2447, 0.5105, 0.2447]
])
expected_out = torch.tensor([
    [1.0000, 2.0000],
    [2.3402, 3.3404],
    [3.0000, 4.0000]
])

assert torch.allclose(weights_hand.squeeze(), expected_weights, atol=1e-3)
assert torch.allclose(out_hand.squeeze(), expected_out, atol=1e-3)
print("Attention Unit Tests & Hand Example Verification PASSED!")



## 2. Multi-Head Attention & Encoder Block


In [1]:
# Task 2: Multi-Head Attention and Encoder Block
class MyMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        out, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_o(out), attn_weights

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.mha = MyMultiHeadAttention(d_model, n_heads)
        self.dropout1 = nn.Dropout(dropout)
        
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LN Architecture
        norm_x = self.ln1(x)
        attn_out, weights = self.mha(norm_x, mask)
        x = x + self.dropout1(attn_out)
        
        norm_x2 = self.ln2(x)
        ffn_out = self.ffn(norm_x2)
        x = x + self.dropout2(ffn_out)
        return x, weights

# Parameter Reconciliation Verification for d=128, d_ff=512
block = TransformerEncoderBlock(d_model=128, n_heads=4, d_ff=512)
param_count = sum(p.numel() for p in block.parameters())
# Derivation:
# Weights: 4 * d^2 (MHA) + 2 * d * d_ff (FFN) = 4*(128^2) + 2*128*512 = 65536 + 131072 = 196,608
# Biases & LN: MHA biases (4*d + d = 640), FFN biases (d_ff + d = 640), LN1 params (2*d = 256), LN2 params (2*d = 256)
# Total = 196,608 + 640 + 640 + 256 + 256 = 198,400.
print(f"Total Block Parameters: {param_count}")
assert param_count == 198400
print("Parameter count reconciliation successfully verified!")

